In [1]:
import os
from transformers import RobertaTokenizer
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import torch.optim as optim


class Autoencoder(nn.Module):
    def __init__(self, device):
        super(Autoencoder, self).__init__()
        self.device = device
        self.encoder = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(128, 64)
        ).to(self.device)

        self.decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 512)
        ).to(self.device)

    def forward(self, x):
        x = x.to(self.device)  # Ensure the input is on the right device
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded


# Check if CUDA is available
print("CUDA available: ", torch.cuda.is_available())

# If CUDA is available, print the GPU device name
if torch.cuda.is_available():
    print("Device name:", torch.cuda.get_device_name(0))

# Check if GPU is available and set device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the tokenizer
tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")


# Function to load and tokenize the code files
def load_and_tokenize_c_files(folder_path):
    tokens = []
    for filename in os.listdir(folder_path):
        if filename.endswith('.c'):
            file_path = os.path.join(folder_path, filename)
            # Open the file with UTF-8 encoding and error handling
            with open(file_path, 'r', encoding='utf-8', errors='replace') as file:
                code = file.read()
                # Tokenize the code
                encoded = tokenizer.encode_plus(
                    code,
                    add_special_tokens=True,
                    max_length=512,
                    padding='max_length',
                    truncation=True,
                    return_tensors='pt'
                )
                tokens.append(encoded['input_ids'])
    return tokens


# Example usage for AI-generated code
tokens = load_and_tokenize_c_files("C:/Users/ABHIMANYU/PycharmProjects/cypherhackathon/code_sample_public/3_GPT-code/")

# Instantiate the model
autoencoder = Autoencoder(device)

# Create a DataLoader for the tokenized AI-generated data
ai_tokens = torch.cat(tokens)  # Concatenate token tensors
# Split the data into training and validation sets
train_data, val_data = train_test_split(ai_tokens, test_size=0.2)

# Create DataLoaders for both sets
train_loader = DataLoader(TensorDataset(train_data), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(val_data), batch_size=16, shuffle=False)

# Define optimizer and loss function
optimizer = optim.Adam(autoencoder.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
loss_fn = nn.MSELoss()

# Training loop
autoencoder.train()
for epoch in range(20):  # Adjust based on your needs
    for batch in train_loader:
        inputs = batch[0].to(device).float()
        optimizer.zero_grad()
        outputs = autoencoder(inputs)
        loss = loss_fn(outputs, inputs)  # Minimize reconstruction error
        loss.backward()
        optimizer.step()

    # Adjust learning rate based on the scheduler
    scheduler.step()
    # Optionally, track learning rate
    # current_lr = scheduler.get_last_lr()

    # Evaluate on validation set
    val_loss = 0.0
    with torch.no_grad():
        for val_batch in val_loader:
            val_inputs = val_batch[0].to(device).float()
            val_outputs = autoencoder(val_inputs)
            val_loss += loss_fn(val_outputs, val_inputs).item()
    print(f"Epoch {epoch + 1}, Train Loss: {loss.item()}, Validation Loss: {val_loss}")


def detect_anomalies(code_sample):
    tokenized = tokenizer.encode_plus(
        code_sample,
        add_special_tokens=True,
        max_length=512,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )['input_ids'].to(device)

    with torch.no_grad():
        reconstructed = autoencoder(tokenized.float())
        loss = loss_fn(reconstructed, tokenized.float())  # Reconstruction error
        return loss.item()


# Example usage with a human-written code sample
human_code_sample = '''
#include <stdio.h>

void two_sum(int* nums, int numsSize, int target, int* returnSize, int* result) {
    for (int i = 0; i < numsSize; i++) {
        for (int j = i + 1; j < numsSize; j++) {
            if (nums[i] + nums[j] == target) {
                result[0] = i;  // First index
                result[1] = j;  // Second index
                *returnSize = 2; // Size of the result
                return;          // Return once the solution is found
            }
        }
    }
    *returnSize = 0; // In case no solution is found
}

int main() {
    int nums[] = {2, 7, 11, 15};
    int target = 9;
    int returnSize;
    int result[2];
    
    two_sum(nums, sizeof(nums) / sizeof(nums[0]), target, &returnSize, result);
    
    if (returnSize == 2) {
        printf("Indices: [%d, %d]\n", result[0], result[1]); // Output: Indices: [0, 1]
    } else {
        printf("No solution found.\n");
    }

    return 0;
}

'''

anomaly_score = detect_anomalies(human_code_sample)
print(f"Anomaly Score: {anomaly_score}")

# You can set a threshold for the anomaly score to classify the code
threshold = 0.05  # Example threshold, adjust based on validation results

if anomaly_score > threshold:
    print("Detected as Human-Written Code")
else:
    print("Detected as AI-Generated Code")


CUDA available:  True
Device name: NVIDIA GeForce RTX 3070 Ti Laptop GPU
Epoch 1, Train Loss: 219691392.0, Validation Loss: 57721037312.0
Epoch 2, Train Loss: 255576608.0, Validation Loss: 57251632048.0
Epoch 3, Train Loss: 264891680.0, Validation Loss: 57698395504.0
Epoch 4, Train Loss: 259400416.0, Validation Loss: 56771824288.0
Epoch 5, Train Loss: 263914720.0, Validation Loss: 56779763760.0
Epoch 6, Train Loss: 194122784.0, Validation Loss: 56362484384.0
Epoch 7, Train Loss: 238614304.0, Validation Loss: 56301941200.0
Epoch 8, Train Loss: 205388000.0, Validation Loss: 56134340336.0
Epoch 9, Train Loss: 240942592.0, Validation Loss: 56096653408.0
Epoch 10, Train Loss: 231885536.0, Validation Loss: 56050722080.0
Epoch 11, Train Loss: 204734512.0, Validation Loss: 55647411264.0
Epoch 12, Train Loss: 244516944.0, Validation Loss: 55600040064.0
Epoch 13, Train Loss: 250622384.0, Validation Loss: 55550596608.0
Epoch 14, Train Loss: 256092736.0, Validation Loss: 55586938816.0
Epoch 15, Tr